[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-code/03-hooks-automatizacion.ipynb)

# Hooks y automatización — Notebook interactivo

Los hooks de Claude Code son comandos shell que se ejecutan en eventos del ciclo de vida.
Aquí simulamos su comportamiento y construimos las funciones que irían dentro de los scripts.

In [ ]:
import anthropic
import subprocess
import json
import os
import re
from pathlib import Path
from datetime import datetime

client = anthropic.Anthropic()
print("Cliente listo")

## 1. Simular el sistema de hooks

In [ ]:
# Registro de hooks (equivalente al settings.json)
HOOKS_CONFIG = {
    "PreToolUse": {
        "Edit": [],      # Se ejecuta ANTES de que Claude edite un archivo
        "Bash": [],      # Se ejecuta ANTES de que Claude ejecute un comando
    },
    "PostToolUse": {
        "Edit": [],      # Se ejecuta DESPUÉS de que Claude edite un archivo
        "Bash": [],
    },
    "Notification": [],  # Cuando Claude envía una notificación
    "Stop": [],          # Cuando termina la sesión
}

AUDIT_LOG = []  # Log de acciones de Claude

def registrar_hook(evento: str, subtipo: str, funcion):
    """Registra un hook para un evento."""
    if subtipo and subtipo in HOOKS_CONFIG.get(evento, {}):
        HOOKS_CONFIG[evento][subtipo].append(funcion)
    elif evento in HOOKS_CONFIG and isinstance(HOOKS_CONFIG[evento], list):
        HOOKS_CONFIG[evento].append(funcion)

def disparar_hook(evento: str, subtipo: str = None, contexto: dict = None) -> bool:
    """Dispara todos los hooks registrados para un evento. Retorna False si alguno bloquea."""
    hooks = []
    if subtipo and subtipo in HOOKS_CONFIG.get(evento, {}):
        hooks = HOOKS_CONFIG[evento][subtipo]
    elif isinstance(HOOKS_CONFIG.get(evento), list):
        hooks = HOOKS_CONFIG[evento]
    
    for hook in hooks:
        resultado = hook(contexto or {})
        if resultado is False:  # Hook bloqueante
            return False
    return True

print("Sistema de hooks inicializado")

## 2. Hook de auditoría de comandos Bash

In [ ]:
COMANDOS_PROHIBIDOS = [
    r"rm\s+-rf",
    r"DROP\s+TABLE",
    r"git\s+push\s+--force",
    r"curl.*\|.*sh",
    r"wget.*\|.*sh",
]

def hook_auditoria_bash(ctx: dict) -> bool:
    """PreToolUse(Bash): registra y valida comandos antes de ejecutarlos."""
    comando = ctx.get("command", "")
    timestamp = datetime.now().isoformat()
    
    # Registrar en audit log
    AUDIT_LOG.append({"timestamp": timestamp, "comando": comando, "estado": "pendiente"})
    
    # Verificar si es peligroso
    for patron in COMANDOS_PROHIBIDOS:
        if re.search(patron, comando, re.IGNORECASE):
            print(f"  🚫 BLOQUEADO: '{comando}' — coincide con patrón prohibido: {patron}")
            AUDIT_LOG[-1]["estado"] = "bloqueado"
            return False  # Bloquear el comando
    
    print(f"  ✅ PERMITIDO: '{comando}'")
    AUDIT_LOG[-1]["estado"] = "permitido"
    return True

registrar_hook("PreToolUse", "Bash", hook_auditoria_bash)

# Simular comandos que Claude intentaría ejecutar
COMANDOS_PRUEBA = [
    "pytest tests/ -v",
    "git add -A && git commit -m 'feat: nueva feature'",
    "rm -rf node_modules",
    "git push --force origin main",
    "ruff check . --fix",
]

print("Simulando PreToolUse(Bash) hooks:\n")
for cmd in COMANDOS_PRUEBA:
    permitido = disparar_hook("PreToolUse", "Bash", {"command": cmd})

print(f"\nAudit log: {len(AUDIT_LOG)} comandos")
print(f"Bloqueados: {sum(1 for e in AUDIT_LOG if e['estado'] == 'bloqueado')}")
print(f"Permitidos: {sum(1 for e in AUDIT_LOG if e['estado'] == 'permitido')}")

## 3. Hook de linting automático tras edición

In [ ]:
import tempfile

def hook_lint_post_edit(ctx: dict):
    """PostToolUse(Edit): ejecuta linting tras cada edición de Claude."""
    archivo = ctx.get("file_path", "")
    
    if archivo.endswith(".py"):
        # En producción: subprocess.run(["ruff", "check", archivo, "--fix"])
        print(f"  🔍 [ruff] Verificando {archivo}...")
        # Simular resultado del linter
        tiene_errores = "legacy" in archivo or "old" in archivo
        if tiene_errores:
            print(f"  ⚠️  [ruff] Encontrados problemas en {archivo} — corrigiendo...")
        else:
            print(f"  ✅ [ruff] {archivo} — sin problemas")
    
    elif archivo.endswith((".ts", ".tsx")):
        print(f"  🔍 [eslint] Verificando {archivo}...")
        print(f"  ✅ [eslint] {archivo} — sin problemas")
    
    elif archivo.endswith(".json"):
        print(f"  🔍 Validando JSON: {archivo}...")
        print(f"  ✅ JSON válido")

registrar_hook("PostToolUse", "Edit", hook_lint_post_edit)

# Simular que Claude edita varios archivos
ARCHIVOS_EDITADOS = [
    "src/services/payment.py",
    "src/legacy/old_auth.py",
    "src/components/Button.tsx",
    "config/settings.json",
]

print("Simulando PostToolUse(Edit) hooks:\n")
for archivo in ARCHIVOS_EDITADOS:
    print(f"Claude editó: {archivo}")
    disparar_hook("PostToolUse", "Edit", {"file_path": archivo})
    print()

## 4. Generar mensaje de commit con Claude (Stop hook)

In [ ]:
def generar_mensaje_commit(archivos_modificados: list[str], descripcion: str = "") -> str:
    """Genera un mensaje de commit semántico con Claude."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": f"""Genera un mensaje de commit en inglés siguiendo Conventional Commits.
Formato: type(scope): descripción breve (máx 72 chars)
Types: feat, fix, docs, refactor, test, chore, style

Archivos modificados: {', '.join(archivos_modificados)}
Descripción del cambio: {descripcion or 'Cambios de Claude Code'}

Solo el mensaje, sin explicación."""}]
    )
    return resp.content[0].text.strip()

def hook_auto_commit_sugerido(ctx: dict):
    """Stop hook: sugiere un commit al terminar la sesión."""
    archivos = ctx.get("archivos_modificados", [])
    if not archivos:
        return
    
    print("\n[Stop Hook] Sesión terminada. Archivos modificados:")
    for f in archivos:
        print(f"  - {f}")
    
    mensaje = generar_mensaje_commit(archivos)
    print(f"\n💡 Commit sugerido:")
    print(f"   git commit -m \"{mensaje}\"")

registrar_hook("Stop", None, hook_auto_commit_sugerido)

# Simular fin de sesión
print("Fin de sesión Claude Code:")
disparar_hook("Stop", contexto={
    "archivos_modificados": ARCHIVOS_EDITADOS
})

## 5. Generar configuración de hooks lista para usar

In [ ]:
SETTINGS_JSON = {
    "model": "claude-sonnet-4-6",
    "hooks": {
        "PreToolUse": [
            {
                "matcher": "Bash",
                "hooks": [{"type": "command", "command": "bash scripts/audit_bash.sh"}]
            }
        ],
        "PostToolUse": [
            {
                "matcher": "Edit",
                "hooks": [
                    {"type": "command", "command": "bash scripts/lint_on_edit.sh"},
                    {"type": "command", "command": "bash scripts/run_tests.sh"}
                ]
            }
        ],
        "Notification": [
            {
                "matcher": ".*",
                "hooks": [{"type": "command", "command": "bash scripts/notify.sh"}]
            }
        ],
        "Stop": [
            {
                "matcher": ".*",
                "hooks": [{"type": "command", "command": "bash scripts/auto_commit.sh"}]
            }
        ]
    },
    "permissions": {
        "allow": ["Bash(git:*)", "Bash(pytest:*)", "Bash(ruff:*)", "Bash(npm run:*)"],
        "deny": ["Bash(rm -rf:*)", "Bash(git push --force:*)"]
    }
}

print(".claude/settings.json recomendado:")
print(json.dumps(SETTINGS_JSON, indent=2, ensure_ascii=False))

# Guardar para referencia
Path("/tmp/claude_settings_ejemplo.json").write_text(json.dumps(SETTINGS_JSON, indent=2))
print("\nGuardado en /tmp/claude_settings_ejemplo.json")